# Corporate Credit Rating Transitions with Economic Regime and Momentum-Dependent Markov Models

### Rearch Question
How do economic conditions and recent rating momentum affect
simulated credit-rating transitions, default probabilities, and time to default, compared with a
baseline Markov model?

#### Credit rating state space and assumptions
Annual rating states: AAA, AA, A, BBB, BB, B, CCC, D. Default is absorbing.

#### Overview of the four models
Four models: baseline, economic regime, momentum, and combined.

In [1]:
# Import external libraries
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

In [3]:
# Import custom Python functions
# Resolve imports whether Jupyter was launched from the root.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src").is_dir() and (p / "data").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the project repository.")
for folder in [ROOT, ROOT / "src"]:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

# Previously created modules
from src.fred_loader import load_annual_regimes
from src.regime_multipliers import (
    load_rating_activity, merge_historical_data, estimate_regime_multipliers
)
from src.regime_model import load_baseline_matrix, build_regime_matrices
from src.momentum_multipliers import (
    load_momentum_matrices, estimate_momentum_multipliers
)
from src.momentum_model import build_momentum_matrices

# Supporting modules supplied with this notebook
from src.combined_model import (
    RATINGS, CATEGORIES, REGIMES, MOMENTA, matrix_frame, canonical_table,
    combine_multipliers, build_combined_matrices
)
from src.simulation import (
    annual_regime_series, estimate_regime_transition, build_chain,
    initial_index, simulate_paths
)
from src.metrics import (
    validate_matrix, fixed_default_probabilities, default_curve,
    expected_times, empirical_curve, time_to_default
)
from src.visualization import plot_matrices, plot_paths


FileNotFoundError: Open this notebook inside the project repository.

In [ ]:
# Define constants, random seed and file paths
RANDOM_SEED = 42
N_SIMULATIONS = 10_000
TIME_HORIZON = 20
HORIZONS = [1, 5, 10, 20]
INITIAL_RATING = "BB"
INITIAL_REGIME = "expansion"
INITIAL_MOMENTUM = "flat"
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

# Resolve the two alternative Bangia baseline filenames discussed earlier.
def locate(*candidates):
    for p in candidates:
        if p.is_file():
            return p
    raise FileNotFoundError("Missing data file. Tried: " + ", ".join(map(str, candidates)))

BASELINE_PATH = locate(PROCESSED / "baseline_matrix.csv")
ACTIVITY_PATH = locate(RAW / "annual_rating_activity.csv",
                       RAW / "sp_global" / "annual_rating_activity.csv")
BANGIA_PATHS = [
    locate(RAW / "bangia_unconditional.csv", RAW / "bangia_baseline.csv",
           RAW / "bangia" / "bangia_unconditional.csv", RAW / "bangia" / "bangia_baseline.csv"),
    *[locate(RAW / f"bangia_{m}.csv", RAW / "bangia" / f"bangia_{m}.csv")
      for m in MOMENTA]
]
print("Project root:", ROOT)


#### Our data sources
Use the S&P baseline and annual rating activity, FRED USREC, and Bangia et al. conditional and unconditional transition matrices.

In [ ]:
# Load S&P baseline transition matrix
# Load the S&P baseline already constructed earlier in the project.
baseline_raw = load_baseline_matrix(str(BASELINE_PATH))
display(baseline_raw)

In [ ]:
# Load S&P annual rating activity
activity = load_rating_activity(str(ACTIVITY_PATH))
display(activity.head())

In [ ]:
# Retrieve FRED recession data
# Match the macroeconomic sample to the available rating-activity years.
years = pd.to_numeric(activity["year"], errors="raise").astype(int)
start_year, end_year = int(years.min()), int(years.max())
regimes = load_annual_regimes(start_year, end_year)
display(regimes.head())

In [ ]:
# Merge annual rating activity and economic regimes
merged_data = merge_historical_data(activity, regimes)
display(merged_data.head())
print("Matched observations:", len(merged_data))

In [ ]:
# Load the four Bangia matrices
# Order: unconditional, upward, unchanged, downward.
historical_matrices = load_momentum_matrices(*map(str, BANGIA_PATHS))
for label, matrix in historical_matrices.items():
    print(label, matrix.shape)

In [ ]:
# Validate and inspect datasets
assert set(["year", "upgrade", "unchanged", "downgrade", "default"]).issubset(activity.columns)
assert not activity["year"].duplicated().any(), "Duplicate activity years"
assert not merged_data.empty, "No matched FRED/S&P observations"
assert not merged_data["year"].duplicated().any(), "Duplicate merged years"
assert set(merged_data["year"].astype(int)) == set(years), "Some S&P years did not match FRED"
regime_series = annual_regime_series(regimes)
assert set(regime_series.unique()) == set(REGIMES), "Both regimes are required"
for label, matrix in historical_matrices.items():
    a = np.asarray(matrix, dtype=float)
    assert np.isfinite(a).all() and (a >= 0).all(), label
    # Published Bangia tables are rounded, and need not contain the same rows.
    assert np.allclose(a.sum(axis=1), 1, atol=0.001, rtol=0), label
print("Input checks passed.")
display(regime_series.value_counts().rename("years"))

#### Data quality and limitations

#### Mathematical formulation
$P_{ij}=\Pr(R_{t+1}=j\mid R_t=i)$.

In [ ]:
# Cell 17 — Construct baseline transition matrix
# Existing S&P loader returns annual probabilities; no additional fitting here.
P = matrix_frame(baseline_raw)
P

In [ ]:
# Validate transition matrix
validate_matrix(P)
print("Baseline matrix is stochastic and D is absorbing.")

In [ ]:
# Display transition matrix
display(P.style.format("{:.2%}"))

In [ ]:
# Plot transition matrix heatmap
plot_matrices({"Baseline": P})

In [ ]:
# Calculate multi-year default probabilities
baseline_pd = fixed_default_probabilities(P, HORIZONS)
baseline_pd.columns = [f"{h}-year" for h in HORIZONS]
display(baseline_pd.style.format("{:.2%}"))

In [ ]:
# Calculate expected time to default
baseline_chain = build_chain(P)
baseline_times = pd.Series(
    expected_times(baseline_chain["K"], baseline_chain["default_mask"]),
    index=RATINGS, name="Expected years to default"
)
display(baseline_times.to_frame().round(2))

#### Interpret baseline results

#### Defining economic regimes
Expansion and recession are annual regimes. Under the existing FRED loader, a year with at least one recession month is classified as recession.

In [ ]:
# Visualize historical recession periods
fig, ax = plt.subplots(figsize=(12, 2.6))
ax.bar(regime_series.index, regime_series.eq("recession").astype(int), width=1)
ax.set(yticks=[0, 1], yticklabels=["Expansion", "Recession"],
       xlabel="Year", title="Annual US economic regimes (FRED USREC)")
plt.show()

#### Economic multiplier estimation
We estimate regime-specific category multipliers by comparing historical rating activity by regime against the unconditional reference.

In [ ]:
# Calculate the 8 economic multipliers
# Eight multipliers estimated from annual S&P activity joined to USREC.
alpha_raw = estimate_regime_multipliers(merged_data)
alpha = canonical_table(alpha_raw, REGIMES)

#### Adjusted transition probabilities
$P_{ij}^{(e)}=P_{ij}\alpha_{e,c(i,j)}/\sum_k P_{ik}\alpha_{e,c(i,k)}$.

In [ ]:
# Construct expansion and recession matrices
# Use the original estimator output for compatibility with your existing module.
regime_matrices = {str(k).lower(): matrix_frame(v)
                   for k, v in build_regime_matrices(P, alpha_raw).items()}

In [ ]:
# Validate both matrices
assert set(regime_matrices) == set(REGIMES)
for e, matrix in regime_matrices.items():
    validate_matrix(matrix)
    print(e, "passed")

In [ ]:
# Plot both matrix heatmaps
plot_matrices(regime_matrices)

In [ ]:
# Plot both matrix heatmaps
plot_matrices(regime_matrices)

In [ ]:
# Calculate regime-specific default probabilities
# Conditional scenario: keep the chosen economic regime fixed for all years.
regime_pd = pd.concat({e: fixed_default_probabilities(M, HORIZONS)
                       for e, M in regime_matrices.items()}, axis=1)
display(regime_pd.style.format("{:.2%}"))

#### Interpret regime effects

#### Defining rating momentum
Momentum records whether the preceding rating movement was upward, unchanged, or downward.

#### Introduce Bangia et al.'s research
The conditional transition matrices are extracted from Bangia et al., Tables 4.2 and 4.3.

In [ ]:
# Inspect extracted historical transition matrices
for label, matrix in historical_matrices.items():
    print(f"Bangia: {label}")
    display(pd.DataFrame(matrix).round(4))

#### Momentum multiplier estimation
 We compare average conditional transition-category probabilities with the unconditional reference using the existing momentum estimator.

In [ ]:
# Calculate the 12 momentum multipliers
# Twelve multipliers estimated from Bangia's historical conditional matrices.
beta_raw = estimate_momentum_multipliers(historical_matrices)
beta = canonical_table(beta_raw, MOMENTA)

In [ ]:
# Display multiplier table
display(beta.loc[["down", "flat", "up"]].T.style.format("{:.3f}"))
# Compare the estimates to the previously extracted literature values.
reference = pd.DataFrame(
    [[.898, .974, 1.266, 1.772], [1.127, .992, .991, 1.111],
     [1.158, 1.026, .650, .386]],
    index=["down", "flat", "up"], columns=CATEGORIES
)
display((beta - reference).round(4))

#### Mathematical formulation
$P_{ij}^{(m)}=P_{ij}\beta_{m,c(i,j)}/\sum_k P_{ik}\beta_{m,c(i,k)}$.

In [ ]:
# Construct three momentum matrices
momentum_matrices = {str(k).lower(): matrix_frame(v)
                     for k, v in build_momentum_matrices(P, beta_raw).items()}

In [ ]:
# Validate transition matrices
assert set(momentum_matrices) == set(MOMENTA)
for m, matrix in momentum_matrices.items():
    validate_matrix(matrix)
    print(m, "passed")

In [ ]:
# Plot momentum matrix heatmaps
plot_matrices(momentum_matrices)

In [ ]:
# Calculate momentum-specific default probabilities
# Momentum is an additional state; it changes after every transition.
momentum_chain = build_chain(P, momentum_matrices=momentum_matrices)
momentum_pd = pd.DataFrame({
    (m, h): [default_curve(momentum_chain["K"],
                           initial_index(momentum_chain, r, momentum=m),
                           momentum_chain["default_mask"], h)[-1] for r in RATINGS]
    for m in MOMENTA for h in HORIZONS
}, index=RATINGS)
display(momentum_pd.style.format("{:.2%}"))

#### Interpret momentum effects

#### Combined model  and assumptions
$P_{ij}^{(e,m)}=P_{ij}\alpha_{e,c(i,j)}\beta_{m,c(i,j)}/\sum_kP_{ik}\alpha_{e,c(i,k)}\beta_{m,c(i,k)}$.


 We assume separability of economic-regime and momentum effects. This is a modeling assumption, not an empirically estimated interaction.

In [ ]:
# Calculate 24 combined multipliers
# Multiplicative separability: gamma[e,m,c] = alpha[e,c] * beta[m,c].
combined_multipliers = combine_multipliers(alpha, beta)
assert combined_multipliers.shape == (6, 4)

In [ ]:
# Display combined multiplier table
display(combined_multipliers.style.format("{:.3f}"))

In [ ]:
# Construct six transition matrices
combined_matrices = build_combined_matrices(P, combined_multipliers)

In [ ]:
# Validate all six matrices
assert set(combined_matrices) == {(e, m) for e in REGIMES for m in MOMENTA}
for (e, m), matrix in combined_matrices.items():
    validate_matrix(matrix)
    print(e, m, "passed")

In [ ]:
# Visualize all six matrices
plot_matrices({f"{e.title()} / {m}": combined_matrices[e, m]
               for e in REGIMES for m in MOMENTA})

In [ ]:
# Calculate conditional default probabilities
# One-year default probabilities for each initial rating/regime/momentum.
combined_pd = pd.DataFrame({(e, m): M["D"]
                            for (e, m), M in combined_matrices.items()})
display(combined_pd.style.format("{:.2%}"))

#### Interpret combined effects

#### Monte Carlo methodology

We simulate annual transitions from the four models. The economic regime evolves according to a separately estimated two-state chain; momentum updates after each nondefault transition. Conditional on the current regime, the next regime is sampled independently of the rating transition.

In [ ]:
# Configure simulation parameters
# Estimate an annual two-state Markov chain for the economic regime.
G = estimate_regime_transition(regimes)
print("Economic regime transition matrix")
display(G.style.format("{:.2%}"))

chains = {
    "Baseline": baseline_chain,
    "Regime": build_chain(P, regime_matrices=regime_matrices, G=G),
    "Momentum": momentum_chain,
    "Combined": build_chain(P, combined_matrices=combined_matrices, G=G)
}
initials = {name: initial_index(chain, INITIAL_RATING, INITIAL_REGIME, INITIAL_MOMENTUM)
            for name, chain in chains.items()}
paths = {}
# Independent reproducible random streams for all four models.
seeds = dict(zip(chains, np.random.SeedSequence(RANDOM_SEED).spawn(len(chains))))

In [ ]:
# Simulate baseline model
paths["Baseline"] = simulate_paths(
    chains["Baseline"], initials["Baseline"], N_SIMULATIONS, TIME_HORIZON, seeds["Baseline"]
)
print(paths["Baseline"].shape)  # (simulated firms, years including year 0)

In [ ]:
# Simulate economic regime model
paths["Regime"] = simulate_paths(
    chains["Regime"], initials["Regime"], N_SIMULATIONS, TIME_HORIZON, seeds["Regime"]
)
print(paths["Regime"].shape)  # (simulated firms, years including year 0)

In [ ]:
# Simulate momentum model
paths["Momentum"] = simulate_paths(
    chains["Momentum"], initials["Momentum"], N_SIMULATIONS, TIME_HORIZON, seeds["Momentum"]
)
print(paths["Momentum"].shape)  # (simulated firms, years including year 0)

In [ ]:
# Simulate combined model
paths["Combined"] = simulate_paths(
    chains["Combined"], initials["Combined"], N_SIMULATIONS, TIME_HORIZON, seeds["Combined"]
)
print(paths["Combined"].shape)  # (simulated firms, years including year 0)

In [ ]:
# Plot sample credit rating trajectories
plot_paths(paths, n_paths=5)

In [ ]:
# Calculate empirical default probabilities
empirical = pd.DataFrame({name: empirical_curve(p) for name, p in paths.items()})
empirical.index.name = "Year"
display(empirical.loc[HORIZONS].style.format("{:.2%}"))

In [ ]:
# Analyze time to default
default_times = {name: time_to_default(p) for name, p in paths.items()}
# Nondefaults are right-censored; do not treat them as zero or discard them
# when estimating the restricted mean E[min(T_D, H)].
censoring = pd.DataFrame({name: {
    "Defaults observed": int(np.isfinite(t).sum()),
    "Censored paths": int(np.isinf(t).sum()),
    "Restricted mean years": np.minimum(t, TIME_HORIZON).mean()
} for name, t in default_times.items()}).T
display(censoring.round(3))

#### Interpret simulation results

#### Comparison metrics
We compare cumulative default probabilities, exact mean time to default, and restricted means using identical initial conditions. Predictive likelihood and AIC require shared observed transition data not yet available.

In [ ]:
# Compare one-year default probabilities
# Exact probabilities from the appropriately augmented transition matrix.
analytical = pd.DataFrame({name: default_curve(chain["K"], initials[name],
                                              chain["default_mask"], TIME_HORIZON)
                           for name, chain in chains.items()})
analytical.index.name = "Year"
display(analytical.loc[[1]].T.rename(columns={1: "1-year PD"}).style.format("{:.2%}"))

In [ ]:
# Compare five-year default probabilities
display(analytical.loc[[5]].T.rename(columns={5: "5-year PD"}).style.format("{:.2%}"))

In [ ]:
# Compare ten-year default probabilities
display(analytical.loc[[10]].T.rename(columns={10: "10-year PD"}).style.format("{:.2%}"))

In [ ]:
# Compare expected time to default
exact_times = pd.Series({name: expected_times(chain["K"], chain["default_mask"])[initials[name]]
                         for name, chain in chains.items()}, name="Expected years to default")
# Compare restricted means over the same finite horizon as simulations.
restricted_exact = (1 - analytical.iloc[:-1]).sum(axis=0).rename("Exact restricted mean years")
display(pd.concat([exact_times, restricted_exact, censoring["Restricted mean years"]], axis=1).round(3))

In [ ]:
# Plot cumulative default probability curves
ax = analytical.plot(figsize=(10, 5), linewidth=2)
ax.set(xlabel="Years", ylabel="Cumulative probability of default",
       title=f"Cumulative default probability: {INITIAL_RATING}", ylim=(0, 1))
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
plt.show()

In [ ]:
# Compare analytical and simulated results
# Monte Carlo sampling error is measured against the exact model probabilities.
comparison = pd.concat({"Analytical": analytical.loc[HORIZONS],
                        "Simulated": empirical.loc[HORIZONS],
                        "Absolute error": (analytical - empirical).abs().loc[HORIZONS]}, axis=1)
display(comparison.style.format("{:.3%}"))
ax = (empirical - analytical).plot(figsize=(10, 4))
ax.axhline(0, color="black", linestyle="--", linewidth=1)
ax.set(xlabel="Years", ylabel="Simulated minus analytical PD")
plt.show()

In [ ]:
# Sensitivity analysis of multipliers
# Sensitivity: power-transform the positive multiplier tables.
# Strength 0 removes both effects; strength 1 is the estimated combined model.
sensitivity = {}
for strength in [0, 0.5, 1, 1.5, 2]:
    weights = combine_multipliers(alpha.pow(strength), beta.pow(strength))
    matrices = build_combined_matrices(P, weights)
    chain = build_chain(P, combined_matrices=matrices, G=G)
    initial = initial_index(chain, INITIAL_RATING, INITIAL_REGIME, INITIAL_MOMENTUM)
    curve = default_curve(chain["K"], initial, chain["default_mask"], TIME_HORIZON)
    sensitivity[strength] = {f"{h}-year PD": curve[h] for h in HORIZONS}
sensitivity_table = pd.DataFrame(sensitivity).T
sensitivity_table.index.name = "Multiplier strength"
display(sensitivity_table.style.format("{:.2%}"))
ax = sensitivity_table.plot(marker="o", figsize=(9, 4))
ax.set(ylabel="Probability of default", xlabel="Multiplier strength")
plt.show()

In [ ]:
# Generate final results summary table
summary = pd.DataFrame({
    "1-year PD": analytical.loc[1],
    "5-year PD": analytical.loc[5],
    "10-year PD": analytical.loc[10],
    "20-year PD": analytical.loc[20],
    "Expected years to default": exact_times,
    "20-year restricted mean (exact)": restricted_exact,
    "20-year restricted mean (MC)": censoring["Restricted mean years"]
})
formats = {c: "{:.2%}" if c.endswith("PD") else "{:.2f}" for c in summary.columns}
display(summary.style.format(formats))
PROCESSED.mkdir(parents=True, exist_ok=True)
summary.to_csv(PROCESSED / "model_comparison.csv")

#### Discuss differences between models

#### Limitations of the baseline model
The baseline assumes homogeneous annual transitions and no dependence on past ratings beyond the current rating.

#### Limitations of economic multipliers
Regime multipliers use aggregate rating activity and a binary annual recession definition. The regime process is assumed time-homogeneous.

#### Limitations of momentum multipliers
Momentum multipliers are literature-derived and may not transfer perfectly across samples, agencies, or time periods.

#### Assumptions of the combined model
The combined model assumes multiplicative separability and does not estimate interactions from jointly labeled firm-level data.

#### Final conclusions

#### Potential extensions
firm-level validation, joint interaction estimation, alternative macro indicators and uncertainty quantification.

#### References
S&P Global Annual Global Corporate Default and Rating Transition Study (exact edition); FRED USREC (https://fred.stlouisfed.org/series/USREC); Bangia et al